In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Colour palette used throughout
STATUS_ORDER  = ['Worsened', 'Stayed the same', 'Improved']
STATUS_COLORS = ['#C0392B', '#F39C12', '#27AE60']

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})
print("✅ Libraries loaded")


## 4. Data Preprocessing

In [ ]:
# The CSV is already cleaned:
#   - financial_status has no leading spaces
#   - monthly_income is numeric (text/blanks replaced with median 5000)
#   - barriers_bank NaN filled with 0 (means no barrier — person has a bank)

# Encode the only two remaining text columns: county and Sex
cat_cols = [c for c in df.select_dtypes(include='object').columns if c != 'financial_status']
print(f"Columns to encode: {cat_cols}")

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le
    print(f"  {col}: {le.classes_[:5].tolist()} ...")

# Encode target
target_le = LabelEncoder()
df['financial_status'] = target_le.fit_transform(df['financial_status'])
print(f"\nTarget encoding: {dict(zip(target_le.classes_, target_le.transform(target_le.classes_)))}")
print(f"Any remaining nulls: {df.isnull().sum().sum()}")
print("✅ Preprocessing complete")


In [ ]:
X = df.drop('financial_status', axis=1)
y = df['financial_status']

# stratify=y keeps class proportions equal in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set : {len(X_train):,} rows (80%)")
print(f"Test set     : {len(X_test):,} rows (20%)")
print(f"\nTest class distribution:")
for cls, count in zip(target_le.classes_, np.bincount(y_test)):
    print(f"  {cls}: {count:,} ({count/len(y_test)*100:.1f}%)")
